# Eval & Observability: Automated Benchmarking

In the previous notebooks, we learned how to evaluate individual responses (Notebook 01), observe agent internals (Notebook 02), and assess reasoning trajectories (Notebook 03).

Now we bring it all together into an **Automated Benchmarking Pipeline** — a repeatable system that:
1. Defines a **Dataset** of test cases with expected outcomes.
2. Runs an agent against every test case.
3. Applies **heuristic + LLM-based evaluation** to each result.
4. Aggregates scores into a **Benchmark Report**.
5. Enables **regression detection** when you change a model or prompt.

This is how production teams ensure their agents don't silently degrade over time.

---

## 1. Environment Setup

In [1]:
import os
import time
import json
from datetime import datetime
from typing import Annotated, TypedDict, List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()
agent_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("Benchmarking Environment Ready!")

/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with cri

Benchmarking Environment Ready!


## 2. Defining the Benchmark Dataset
A good benchmark dataset includes:
- **Input**: The query/task for the agent.
- **Expected Behavior**: What tools should be called, in what order.
- **Expected Output**: A reference answer or key criteria.
- **Metadata**: Difficulty level, category, and tags.

In [2]:
BENCHMARK_DATASET = [
    {
        "id": "TC-001",
        "category": "code_generation",
        "difficulty": "easy",
        "query": "Write a Python function to check if a string is a palindrome.",
        "expected_keywords": ["def", "palindrome", "return"],
        "must_contain_code": True,
        "max_words": 500,
    },
    {
        "id": "TC-002",
        "category": "code_generation",
        "difficulty": "medium",
        "query": "Write a Python function that finds the two numbers in a list that add up to a target sum.",
        "expected_keywords": ["def", "return", "target"],
        "must_contain_code": True,
        "max_words": 800,
    },
    {
        "id": "TC-003",
        "category": "explanation",
        "difficulty": "easy",
        "query": "Explain the difference between a list and a tuple in Python in 3 sentences.",
        "expected_keywords": ["list", "tuple", "mutable"],
        "must_contain_code": False,
        "max_words": 200,
    },
    {
        "id": "TC-004",
        "category": "code_generation",
        "difficulty": "hard",
        "query": "Write a Python decorator that caches function results with a TTL (time-to-live) in seconds.",
        "expected_keywords": ["def", "decorator", "cache", "time"],
        "must_contain_code": True,
        "max_words": 1000,
    },
    {
        "id": "TC-005",
        "category": "debugging",
        "difficulty": "medium",
        "query": "This Python code has a bug. Fix it:\ndef fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-3)",
        "expected_keywords": ["fibonacci", "n-2"],
        "must_contain_code": True,
        "max_words": 500,
    },
]

print(f"Benchmark dataset loaded: {len(BENCHMARK_DATASET)} test cases")
for tc in BENCHMARK_DATASET:
    print(f"  [{tc['id']}] {tc['category']} | {tc['difficulty']} | {tc['query'][:60]}...")

Benchmark dataset loaded: 5 test cases
  [TC-001] code_generation | easy | Write a Python function to check if a string is a palindrome...
  [TC-002] code_generation | medium | Write a Python function that finds the two numbers in a list...
  [TC-003] explanation | easy | Explain the difference between a list and a tuple in Python ...
  [TC-004] code_generation | hard | Write a Python decorator that caches function results with a...
  [TC-005] debugging | medium | This Python code has a bug. Fix it:
def fibonacci(n):
    if...


## 3. Evaluation Functions
We'll reuse and extend the evaluation patterns from Notebook 01.

In [3]:
# --- Heuristic Evaluators ---
def eval_contains_code(response: str) -> bool:
    return "```" in response

def eval_max_words(response: str, max_words: int) -> bool:
    return len(response.split()) <= max_words

def eval_keywords(response: str, keywords: List[str]) -> float:
    """Returns fraction of expected keywords found (0.0 to 1.0)."""
    if not keywords:
        return 1.0
    found = sum(1 for kw in keywords if kw.lower() in response.lower())
    return found / len(keywords)

def eval_no_apology(response: str) -> bool:
    apology_phrases = ["i'm sorry", "i apologize", "sorry about that"]
    return not any(p in response.lower() for p in apology_phrases)

# --- LLM Judge ---
class JudgeResult(BaseModel):
    """Structured evaluation from the LLM judge."""
    correctness: int = Field(description="1-5: Does the response correctly solve the problem?")
    completeness: int = Field(description="1-5: Are edge cases and details addressed?")
    clarity: int = Field(description="1-5: Is the explanation clear and well-structured?")
    overall: int = Field(description="1-5: Overall quality score.")
    feedback: str = Field(description="Brief feedback on strengths and weaknesses.")

judge_structured = judge_llm.with_structured_output(JudgeResult)

def llm_evaluate(query: str, response: str) -> JudgeResult:
    system_msg = SystemMessage(content="""You are a strict expert evaluator.
    Grade the AI's response to the user's request.
    Be critical but fair. A score of 5 means professional, production-quality work.""")
    
    prompt = f"USER REQUEST:\n{query}\n\nAI RESPONSE:\n{response}\n\nEvaluate."
    return judge_structured.invoke([system_msg, HumanMessage(content=prompt)])

print("Evaluation functions ready!")

Evaluation functions ready!


## 4. The Benchmark Runner
This is the core function: it runs the agent against every test case, applies all evaluators, and collects results.

In [4]:
def run_benchmark(dataset: List[dict], llm, run_name: str = "default") -> dict:
    """Run a full benchmark suite and return structured results."""
    results = []
    run_start = time.time()
    
    for i, tc in enumerate(dataset):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(dataset)}] {tc['id']} — {tc['category']} ({tc['difficulty']})")
        print(f"Query: {tc['query'][:80]}...")
        
        # 1. Run the agent
        start = time.time()
        try:
            response = llm.invoke([HumanMessage(content=tc["query"])])
            agent_output = response.content
            latency_ms = round((time.time() - start) * 1000)
            has_error = False
        except Exception as e:
            agent_output = f"ERROR: {str(e)}"
            latency_ms = round((time.time() - start) * 1000)
            has_error = True
        
        # 2. Heuristic evaluation
        h_code = eval_contains_code(agent_output) if tc["must_contain_code"] else True
        h_length = eval_max_words(agent_output, tc["max_words"])
        h_keywords = eval_keywords(agent_output, tc["expected_keywords"])
        h_no_apology = eval_no_apology(agent_output)
        heuristic_pass = h_code and h_length and h_keywords >= 0.5 and h_no_apology
        
        # 3. LLM judge evaluation
        if not has_error:
            judge_result = llm_evaluate(tc["query"], agent_output)
        else:
            judge_result = JudgeResult(correctness=1, completeness=1, clarity=1, overall=1, feedback="Agent errored.")
        
        result = {
            "test_id": tc["id"],
            "category": tc["category"],
            "difficulty": tc["difficulty"],
            "latency_ms": latency_ms,
            "has_error": has_error,
            "heuristic_code": h_code,
            "heuristic_length": h_length,
            "heuristic_keywords": round(h_keywords, 2),
            "heuristic_no_apology": h_no_apology,
            "heuristic_pass": heuristic_pass,
            "judge_correctness": judge_result.correctness,
            "judge_completeness": judge_result.completeness,
            "judge_clarity": judge_result.clarity,
            "judge_overall": judge_result.overall,
            "judge_feedback": judge_result.feedback,
        }
        results.append(result)
        
        print(f"  Latency: {latency_ms}ms | Heuristic: {'✅' if heuristic_pass else '❌'} | Judge: {judge_result.overall}/5")
    
    total_time = round(time.time() - run_start, 1)
    
    return {
        "run_name": run_name,
        "timestamp": datetime.now().isoformat(),
        "total_cases": len(results),
        "total_time_s": total_time,
        "results": results,
    }

print("Benchmark runner ready!")

Benchmark runner ready!


## 5. Running the Benchmark

In [5]:
benchmark_report = run_benchmark(
    dataset=BENCHMARK_DATASET,
    llm=agent_llm,
    run_name="gemini-2.5-flash-baseline"
)


[1/5] TC-001 — code_generation (easy)
Query: Write a Python function to check if a string is a palindrome....
  Latency: 13777ms | Heuristic: ❌ | Judge: 5/5

[2/5] TC-002 — code_generation (medium)
Query: Write a Python function that finds the two numbers in a list that add up to a ta...
  Latency: 18738ms | Heuristic: ✅ | Judge: 5/5

[3/5] TC-003 — explanation (easy)
Query: Explain the difference between a list and a tuple in Python in 3 sentences....
  Latency: 3436ms | Heuristic: ✅ | Judge: 5/5

[4/5] TC-004 — code_generation (hard)
Query: Write a Python decorator that caches function results with a TTL (time-to-live) ...
  Latency: 20344ms | Heuristic: ❌ | Judge: 5/5

[5/5] TC-005 — debugging (medium)
Query: This Python code has a bug. Fix it:
def fibonacci(n):
    if n <= 1:
        ret...
  Latency: 3995ms | Heuristic: ✅ | Judge: 5/5


## 6. Generating the Benchmark Report
Aggregate all results into a clear, actionable report.

In [6]:
def generate_report(report: dict) -> str:
    """Generate a formatted benchmark report."""
    results = report["results"]
    
    # Aggregate scores
    avg_correctness = sum(r["judge_correctness"] for r in results) / len(results)
    avg_completeness = sum(r["judge_completeness"] for r in results) / len(results)
    avg_clarity = sum(r["judge_clarity"] for r in results) / len(results)
    avg_overall = sum(r["judge_overall"] for r in results) / len(results)
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    heuristic_pass_rate = sum(1 for r in results if r["heuristic_pass"]) / len(results) * 100
    error_rate = sum(1 for r in results if r["has_error"]) / len(results) * 100
    
    # By category
    categories = set(r["category"] for r in results)
    by_category = {}
    for cat in categories:
        cat_results = [r for r in results if r["category"] == cat]
        by_category[cat] = round(sum(r["judge_overall"] for r in cat_results) / len(cat_results), 1)
    
    # By difficulty
    difficulties = ["easy", "medium", "hard"]
    by_difficulty = {}
    for diff in difficulties:
        diff_results = [r for r in results if r["difficulty"] == diff]
        if diff_results:
            by_difficulty[diff] = round(sum(r["judge_overall"] for r in diff_results) / len(diff_results), 1)
    
    output = f"""
{'='*60}
  BENCHMARK REPORT: {report['run_name']}
  Timestamp: {report['timestamp']}
  Total Cases: {report['total_cases']} | Total Time: {report['total_time_s']}s
{'='*60}

OVERALL SCORES (1-5)
  Correctness:   {avg_correctness:.1f}/5
  Completeness:  {avg_completeness:.1f}/5
  Clarity:       {avg_clarity:.1f}/5
  Overall:       {avg_overall:.1f}/5

OPERATIONAL METRICS
  Avg Latency:        {avg_latency:.0f}ms
  Heuristic Pass Rate: {heuristic_pass_rate:.0f}%
  Error Rate:          {error_rate:.0f}%

BY CATEGORY"""
    for cat, score in by_category.items():
        output += f"\n  {cat}: {score}/5"
    
    output += "\n\nBY DIFFICULTY"
    for diff, score in by_difficulty.items():
        output += f"\n  {diff}: {score}/5"
    
    output += f"\n\nDETAILED RESULTS"
    output += f"\n{'Test ID':<10} {'Category':<18} {'Diff':<8} {'Latency':<10} {'Heuristic':<10} {'Judge':<8} {'Feedback'}"
    output += f"\n{'-'*100}"
    for r in results:
        h_status = '✅' if r['heuristic_pass'] else '❌'
        output += f"\n{r['test_id']:<10} {r['category']:<18} {r['difficulty']:<8} {r['latency_ms']:<10} {h_status:<10} {r['judge_overall']}/5{'':5} {r['judge_feedback'][:50]}"
    
    output += f"\n{'='*60}"
    return output

report_text = generate_report(benchmark_report)
print(report_text)


  BENCHMARK REPORT: gemini-2.5-flash-baseline
  Timestamp: 2026-04-10T08:41:14.277046
  Total Cases: 5 | Total Time: 90.1s

OVERALL SCORES (1-5)
  Correctness:   5.0/5
  Completeness:  5.0/5
  Clarity:       5.0/5
  Overall:       5.0/5

OPERATIONAL METRICS
  Avg Latency:        12058ms
  Heuristic Pass Rate: 60%
  Error Rate:          0%

BY CATEGORY
  code_generation: 5.0/5
  debugging: 5.0/5
  explanation: 5.0/5

BY DIFFICULTY
  easy: 5.0/5
  medium: 5.0/5
  hard: 5.0/5

DETAILED RESULTS
Test ID    Category           Diff     Latency    Heuristic  Judge    Feedback
----------------------------------------------------------------------------------------------------
TC-001     code_generation    easy     13777      ❌          5/5      The AI provided a perfectly correct, complete, and
TC-002     code_generation    medium   18738      ✅          5/5      The AI provided an excellent solution. The Python 
TC-003     explanation        easy     3436       ✅          5/5      The AI's re

## 7. Regression Detection
When you change a model, update a prompt, or modify agent logic, run the benchmark again and compare reports. Here's a utility to compare two benchmark runs.

In [7]:
def compare_benchmarks(report_a: dict, report_b: dict) -> str:
    """Compare two benchmark reports and flag regressions."""
    results_a = {r["test_id"]: r for r in report_a["results"]}
    results_b = {r["test_id"]: r for r in report_b["results"]}
    
    common_ids = set(results_a.keys()) & set(results_b.keys())
    
    regressions = []
    improvements = []
    unchanged = []
    
    for test_id in sorted(common_ids):
        score_a = results_a[test_id]["judge_overall"]
        score_b = results_b[test_id]["judge_overall"]
        delta = score_b - score_a
        
        entry = {"test_id": test_id, "score_a": score_a, "score_b": score_b, "delta": delta}
        if delta < 0:
            regressions.append(entry)
        elif delta > 0:
            improvements.append(entry)
        else:
            unchanged.append(entry)
    
    output = f"""
{'='*60}
  REGRESSION REPORT
  Comparing: '{report_a['run_name']}' → '{report_b['run_name']}'
{'='*60}

  🔴 Regressions:  {len(regressions)}
  🟢 Improvements: {len(improvements)}
  ⚪ Unchanged:    {len(unchanged)}
"""
    
    if regressions:
        output += "\n⚠️  REGRESSIONS DETECTED:\n"
        for r in regressions:
            output += f"  {r['test_id']}: {r['score_a']}/5 → {r['score_b']}/5 (Δ{r['delta']})\n"
    
    if improvements:
        output += "\n✅ IMPROVEMENTS:\n"
        for r in improvements:
            output += f"  {r['test_id']}: {r['score_a']}/5 → {r['score_b']}/5 (+{r['delta']})\n"
    
    return output

# Simulate a second run (in production, you'd change the model/prompt and re-run)
# For demonstration, we re-run the same benchmark
print("Running comparison benchmark...")
benchmark_report_v2 = run_benchmark(
    dataset=BENCHMARK_DATASET,
    llm=agent_llm,
    run_name="gemini-2.5-flash-v2"
)

comparison = compare_benchmarks(benchmark_report, benchmark_report_v2)
print(comparison)

Running comparison benchmark...

[1/5] TC-001 — code_generation (easy)
Query: Write a Python function to check if a string is a palindrome....
  Latency: 13909ms | Heuristic: ❌ | Judge: 5/5

[2/5] TC-002 — code_generation (medium)
Query: Write a Python function that finds the two numbers in a list that add up to a ta...
  Latency: 18601ms | Heuristic: ✅ | Judge: 5/5

[3/5] TC-003 — explanation (easy)
Query: Explain the difference between a list and a tuple in Python in 3 sentences....
  Latency: 2369ms | Heuristic: ✅ | Judge: 5/5

[4/5] TC-004 — code_generation (hard)
Query: Write a Python decorator that caches function results with a TTL (time-to-live) ...
  Latency: 17771ms | Heuristic: ✅ | Judge: 5/5

[5/5] TC-005 — debugging (medium)
Query: This Python code has a bug. Fix it:
def fibonacci(n):
    if n <= 1:
        ret...
  Latency: 4812ms | Heuristic: ✅ | Judge: 5/5

  REGRESSION REPORT
  Comparing: 'gemini-2.5-flash-baseline' → 'gemini-2.5-flash-v2'

  🔴 Regressions:  0
  🟢 Impr

## Summary

Automated benchmarking is the **final piece** of the production agentic AI puzzle.

| Component | Purpose |
| :--- | :--- |
| **Dataset** | Defines what you're testing — queries, expected outputs, metadata |
| **Heuristic Evaluators** | Fast, free format checks (code blocks, length, keywords) |
| **LLM Judge** | Deep quality assessment (correctness, completeness, clarity) |
| **Benchmark Runner** | Orchestrates agent execution + evaluation for every test case |
| **Report Generator** | Aggregates scores by category, difficulty, and provides detailed per-case results |
| **Regression Detector** | Compares two runs to flag quality drops after changes |

> **Pro-Tip**: Integrate your benchmark into CI/CD. Every pull request that changes a prompt or model should automatically trigger a benchmark run. If the overall score drops below a threshold, block the merge.

---

### 🎉 Module 06 Complete!
You now have the full toolkit to **build**, **observe**, and **evaluate** production-grade agentic systems. The journey from Foundation to Eval & Observability covers the entire lifecycle of an agentic AI application.